<a href="https://colab.research.google.com/github/anjali1947v-bit/AIML_Project_grp28/blob/main/dataset1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:

import requests
import pandas as pd
import time

# Replace this with your contact email
HEADERS = {
    "User-Agent": "StudentResearch anjali1947v@gmail.com",
    "Accept-Encoding": "gzip, deflate",
    "Host": "data.sec.gov"
}

# Selected publicly listed US financial institutions
companies = {
    "JPMorgan Chase": "0000019617",
    "Bank of America": "0000070858",
    "Wells Fargo": "0000072971",
    "Citigroup": "0000831001",
    "Goldman Sachs": "0000886982",
    "Morgan Stanley": "0000895421"
}

# Financial indicators to try collecting
metrics = {
    "TotalAssets": ["Assets"],
    "TotalLiabilities": ["Liabilities"],
    "CashAndCashEquivalents": [
        "CashAndCashEquivalentsAtCarryingValue",
        "CashCashEquivalentsRestrictedCashAndRestrictedCashEquivalents"
    ],
    "Revenue": [
        "RevenueFromContractWithCustomerExcludingAssessedTax",
        "Revenues",
        "SalesRevenueNet"
    ],
    "NetIncome": ["NetIncomeLoss"]
}

session = requests.Session()
session.headers.update(HEADERS)

records = []
errors = []

for company, cik in companies.items():
    url = (
        "https://data.sec.gov/api/xbrl/companyfacts/"
        f"CIK{cik}.json"
    )

    try:
        response = session.get(url, timeout=30)
        response.raise_for_status()
        company_data = response.json()

        facts = company_data.get("facts", {}).get("us-gaap", {})

        for metric_name, possible_tags in metrics.items():
            selected_tag = next(
                (tag for tag in possible_tags if tag in facts),
                None
            )

            if selected_tag is None:
                print(f"Not found: {company} - {metric_name}")
                continue

            units = facts[selected_tag].get("units", {})
            usd_values = units.get("USD", [])

            for item in usd_values:
                if item.get("form") not in ["10-K", "10-Q"]:
                    continue

                if not item.get("end") or "val" not in item:
                    continue

                records.append({
                    "Company": company,
                    "CIK": cik,
                    "Metric": metric_name,
                    "ReportedValueUSD": item["val"],
                    "PeriodStart": item.get("start"),
                    "PeriodEnd": item["end"],
                    "FilingDate": item.get("filed"),
                    "Form": item.get("form"),
                    "FiscalYear": item.get("fy"),
                    "FiscalPeriod": item.get("fp"),
                    "SourceTag": selected_tag,
                    "AccessionNumber": item.get("accn")
                })

        print(f"Collected data for {company}")

    except Exception as e:
        errors.append({"Company": company, "Error": str(e)})
        print(f"Error for {company}: {e}")

    time.sleep(0.2)

df = pd.DataFrame(records)

if not df.empty:
    df = df.drop_duplicates()
    df = df.sort_values(
        ["Company", "Metric", "PeriodEnd", "FilingDate"]
    )
    df.to_csv("sec_financial_data_raw.csv", index=False)

    print("\nCollection completed.")
    print("Rows collected:", len(df))
    print("Companies collected:", df["Company"].nunique())
    print("\nPreview:")
    display(df.head(10))

    print("\nMetrics found:")
    print(df.groupby("Metric").size())

    from google.colab import files
    files.download("sec_financial_data_raw.csv")
else:
    print("No records collected. Check the errors above.")

if errors:
    print("\nCollection errors:")
    display(pd.DataFrame(errors))


Collected data for JPMorgan Chase
Collected data for Bank of America
Collected data for Wells Fargo
Collected data for Citigroup
Not found: Goldman Sachs - Revenue
Collected data for Goldman Sachs
Collected data for Morgan Stanley

Collection completed.
Rows collected: 4797
Companies collected: 6

Preview:


,Company,CIK,Metric,ReportedValueUSD,PeriodStart,PeriodEnd,FilingDate,Form,FiscalYear,FiscalPeriod,SourceTag,AccessionNumber
1006,Bank of America,0000070858,CashAndCashEquivalents,36429000000,None,2006-12-31,2010-02-26,10-K,2009,FY,CashAndCashEquivalentsAtCarryingValue,0001193125-10-041666
1009,Bank of America,0000070858,CashAndCashEquivalents,42531000000,None,2007-12-31,2009-08-07,10-Q,2009,Q2,CashAndCashEquivalentsAtCarryingValue,0001193125-09-168935
1008,Bank of America,0000070858,CashAndCashEquivalents,42531000000,None,2007-12-31,2009-11-06,10-Q,2009,Q3,CashAndCashEquivalentsAtCarryingValue,0001193125-09-227720
1007,Bank of America,0000070858,CashAndCashEquivalents,42531000000,None,2007-12-31,2010-02-26,10-K,2009,FY,CashAndCashEquivalentsAtCarryingValue,0001193125-10-041666
1010,Bank of America,0000070858,CashAndCashEquivalents,42531000000,None,2007-12-31,2011-02-25,10-K,2010,FY,CashAndCashEquivalentsAtCarryingValue,0000950123-11-018743
1011,Bank of America,0000070858,CashAndCashEquivalents,39127000000,None,2008-06-30,2009-08-07,10-Q,2009,Q2,CashAndCashEquivalentsAtCarryingValue,0001193125-09-168935
1012,Bank of America,0000070858,CashAndCashEquivalents,39341000000,None,2008-09-30,2009-11-06,10-Q,2009,Q3,CashAndCashEquivalentsAtCarryingValue,0001193125-09-227720
1016,Bank of America,0000070858,CashAndCashEquivalents,32857000000,None,2008-12-31,2009-08-07,10-Q,2009,Q2,CashAndCashEquivalentsAtCarryingValue,0001193125-09-168935
1015,Bank of America,0000070858,CashAndCashEquivalents,32857000000,None,2008-12-31,2009-11-06,10-Q,2009,Q3,CashAndCashEquivalentsAtCarryingValue,0001193125-09-227720
1014,Bank of America,0000070858,CashAndCashEquivalents,32857000000,None,2008-12-31,2010-02-26,10-K,2009,FY,CashAndCashEquivalentsAtCarryingValue,0001193125-10-041666



Metrics found:
Metric
CashAndCashEquivalents     728
NetIncome                 1530
Revenue                    684
TotalAssets               1019
TotalLiabilities           836
dtype: int64


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [2]:

import pandas as pd

# Load the downloaded SEC dataset
df = pd.read_csv("sec_financial_data_raw.csv")

print("SEC FINANCIAL DATASET ANALYSIS")
print("=" * 40)

print("\n1. Dataset dimensions")
print("Rows:", len(df))
print("Columns:", len(df.columns))

print("\n2. Companies collected")
print(df["Company"].nunique())
print(df["Company"].value_counts())

print("\n3. Financial metrics collected")
print(df["Metric"].value_counts())

print("\n4. Reporting period")
print("Earliest period:", df["PeriodEnd"].min())
print("Latest period:", df["PeriodEnd"].max())

print("\n5. Missing values")
print(df.isnull().sum())

print("\n6. Duplicate rows")
print("Exact duplicate rows:", df.duplicated().sum())

print("\n7. Records per company and metric")
display(
    df.groupby(["Company", "Metric"])
      .size()
      .unstack(fill_value=0)
)

print("\n8. Financial value summary (USD)")
display(
    df.groupby("Metric")["ReportedValueUSD"]
      .agg(["count", "mean", "median", "min", "max"])
      .round(2)
)

print("\n9. Sample records")
display(df.head(10))

# Save a basic analysis report
with open("sec_dataset_analysis_report.txt", "w") as f:
    f.write("SEC Financial Data Collection - Initial Analysis\n")
    f.write("=" * 50 + "\n")
    f.write(f"Rows: {len(df)}\n")
    f.write(f"Columns: {len(df.columns)}\n")
    f.write(f"Companies: {df['Company'].nunique()}\n")
    f.write(f"Metrics: {df['Metric'].nunique()}\n")
    f.write(f"Earliest period: {df['PeriodEnd'].min()}\n")
    f.write(f"Latest period: {df['PeriodEnd'].max()}\n")
    f.write(f"Exact duplicate rows: {df.duplicated().sum()}\n\n")
    f.write("Records per metric:\n")
    f.write(df["Metric"].value_counts().to_string())
    f.write("\n\nMissing values:\n")
    f.write(df.isnull().sum().to_string())

print("\nAnalysis report saved.")


SEC FINANCIAL DATASET ANALYSIS

1. Dataset dimensions
Rows: 4797
Columns: 12

2. Companies collected
6
Company
Bank of America    940
Citigroup          908
Morgan Stanley     812
Goldman Sachs      760
Wells Fargo        691
JPMorgan Chase     686
Name: count, dtype: int64

3. Financial metrics collected
Metric
NetIncome                 1530
TotalAssets               1019
TotalLiabilities           836
CashAndCashEquivalents     728
Revenue                    684
Name: count, dtype: int64

4. Reporting period
Earliest period: 2006-11-30
Latest period: 2026-06-30

5. Missing values
Company                0
CIK                    0
Metric                 0
ReportedValueUSD       0
PeriodStart         2584
PeriodEnd              0
FilingDate             0
Form                   0
FiscalYear             0
FiscalPeriod           0
SourceTag              0
AccessionNumber        0
dtype: int64

6. Duplicate rows
Exact duplicate rows: 0

7. Records per company and metric


Metric,CashAndCashEquivalents,NetIncome,Revenue,TotalAssets,TotalLiabilities
Company,,,,,
Bank of America,168,226,226,184,136
Citigroup,16,311,311,138,132
Goldman Sachs,176,228,0,205,151
JPMorgan Chase,16,225,109,197,139
Morgan Stanley,216,315,1,140,140
Wells Fargo,136,225,37,155,138



8. Financial value summary (USD)


,count,mean,median,min,max
Metric,,,,,
CashAndCashEquivalents,728,1.312681e+11,1.217110e+11,20606000000,461886000000
NetIncome,1530,7.667558e+09,5.114000e+09,-27684000000,58471000000
Revenue,684,4.652325e+10,2.956100e+10,5405000000,182447000000
TotalAssets,1019,1.918431e+12,1.927555e+12,0,5015069000000
TotalLiabilities,836,1.700425e+12,1.701673e+12,607499000000,4640471000000



9. Sample records


,Company,CIK,Metric,ReportedValueUSD,PeriodStart,PeriodEnd,FilingDate,Form,FiscalYear,FiscalPeriod,SourceTag,AccessionNumber
0,Bank of America,70858,CashAndCashEquivalents,36429000000,NaN,2006-12-31,2010-02-26,10-K,2009,FY,CashAndCashEquivalentsAtCarryingValue,0001193125-10-041666
1,Bank of America,70858,CashAndCashEquivalents,42531000000,NaN,2007-12-31,2009-08-07,10-Q,2009,Q2,CashAndCashEquivalentsAtCarryingValue,0001193125-09-168935
2,Bank of America,70858,CashAndCashEquivalents,42531000000,NaN,2007-12-31,2009-11-06,10-Q,2009,Q3,CashAndCashEquivalentsAtCarryingValue,0001193125-09-227720
3,Bank of America,70858,CashAndCashEquivalents,42531000000,NaN,2007-12-31,2010-02-26,10-K,2009,FY,CashAndCashEquivalentsAtCarryingValue,0001193125-10-041666
4,Bank of America,70858,CashAndCashEquivalents,42531000000,NaN,2007-12-31,2011-02-25,10-K,2010,FY,CashAndCashEquivalentsAtCarryingValue,0000950123-11-018743
5,Bank of America,70858,CashAndCashEquivalents,39127000000,NaN,2008-06-30,2009-08-07,10-Q,2009,Q2,CashAndCashEquivalentsAtCarryingValue,0001193125-09-168935
6,Bank of America,70858,CashAndCashEquivalents,39341000000,NaN,2008-09-30,2009-11-06,10-Q,2009,Q3,CashAndCashEquivalentsAtCarryingValue,0001193125-09-227720
7,Bank of America,70858,CashAndCashEquivalents,32857000000,NaN,2008-12-31,2009-08-07,10-Q,2009,Q2,CashAndCashEquivalentsAtCarryingValue,0001193125-09-168935
8,Bank of America,70858,CashAndCashEquivalents,32857000000,NaN,2008-12-31,2009-11-06,10-Q,2009,Q3,CashAndCashEquivalentsAtCarryingValue,0001193125-09-227720
9,Bank of America,70858,CashAndCashEquivalents,32857000000,NaN,2008-12-31,2010-02-26,10-K,2009,FY,CashAndCashEquivalentsAtCarryingValue,0001193125-10-041666



Analysis report saved.


In [3]:

import pandas as pd

df = pd.read_csv("sec_financial_data_raw.csv")

# Keep annual reports only for a more consistent first analysis
annual = df[
    (df["Form"] == "10-K") &
    (df["FiscalPeriod"] == "FY")
].copy()

# Convert dates and values
annual["FilingDate"] = pd.to_datetime(
    annual["FilingDate"], errors="coerce"
)
annual["PeriodEnd"] = pd.to_datetime(
    annual["PeriodEnd"], errors="coerce"
)
annual["ReportedValueUSD"] = pd.to_numeric(
    annual["ReportedValueUSD"], errors="coerce"
)

# For each company, metric and reporting period,
# retain the latest-filed observation
annual = annual.sort_values("FilingDate")

clean = annual.drop_duplicates(
    subset=["Company", "Metric", "PeriodEnd"],
    keep="last"
).copy()

# Sort and save
clean = clean.sort_values(
    ["Company", "PeriodEnd", "Metric"]
)

clean.to_csv("sec_financial_data_clean_annual.csv", index=False)

print("CLEAN ANNUAL DATASET")
print("=" * 35)
print("Rows:", len(clean))
print("Companies:", clean["Company"].nunique())
print("Metrics:", clean["Metric"].nunique())
print("Period range:",
      clean["PeriodEnd"].min(), "to", clean["PeriodEnd"].max())

print("\nRecords per company and metric:")
display(
    clean.groupby(["Company", "Metric"])
         .size()
         .unstack(fill_value=0)
)

print("\nMissing values:")
print(clean.isnull().sum())

print("\nZero total-assets records:")
display(
    clean[
        (clean["Metric"] == "TotalAssets") &
        (clean["ReportedValueUSD"] == 0)
    ][["Company", "PeriodEnd", "FilingDate", "SourceTag"]]
)

print("\nSaved: sec_financial_data_clean_annual.csv")

from google.colab import files
files.download("sec_financial_data_clean_annual.csv")


CLEAN ANNUAL DATASET
Rows: 581
Companies: 6
Metrics: 5
Period range: 2006-11-30 00:00:00 to 2025-12-31 00:00:00

Records per company and metric:


Metric,CashAndCashEquivalents,NetIncome,Revenue,TotalAssets,TotalLiabilities
Company,,,,,
Bank of America,14,19,19,18,18
Citigroup,4,58,58,18,18
Goldman Sachs,16,22,0,20,18
JPMorgan Chase,4,19,19,18,18
Morgan Stanley,21,56,0,18,18
Wells Fargo,11,19,4,18,18



Missing values:
Company               0
CIK                   0
Metric                0
ReportedValueUSD      0
PeriodStart         289
PeriodEnd             0
FilingDate            0
Form                  0
FiscalYear            0
FiscalPeriod          0
SourceTag             0
AccessionNumber       0
dtype: int64

Zero total-assets records:


,Company,PeriodEnd,FilingDate,SourceTag
623,Bank of America,2008-12-31,2011-02-25,Assets



Saved: sec_financial_data_clean_annual.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [4]:
import pandas as pd

raw = pd.read_csv("sec_financial_data_raw.csv")
clean = pd.read_csv("sec_financial_data_clean_annual.csv")

# 1. Check the year coverage for each company and metric
clean["PeriodEnd"] = pd.to_datetime(clean["PeriodEnd"], errors="coerce")
clean["Year"] = clean["PeriodEnd"].dt.year

print("YEAR COVERAGE BY COMPANY AND METRIC")
display(
    clean.groupby(["Company", "Metric"])["Year"]
    .agg(["min", "max", "nunique"])
    .reset_index()
)

# 2. Inspect the SEC tags used for revenue in the raw dataset
print("\nREVENUE SOURCE TAGS IN RAW DATA")
display(
    raw[raw["Metric"] == "Revenue"]
    .groupby(["Company", "SourceTag"])
    .size()
    .reset_index(name="Records")
    .sort_values(["Company", "Records"], ascending=[True, False])
)

# 3. Inspect the zero-assets record
print("\nZERO-ASSETS RECORD IN RAW DATA")
display(
    raw[
        (raw["Metric"] == "TotalAssets") &
        (pd.to_numeric(raw["ReportedValueUSD"], errors="coerce") == 0)
    ][[
        "Company", "ReportedValueUSD", "PeriodStart",
        "PeriodEnd", "FilingDate", "Form",
        "FiscalYear", "FiscalPeriod", "SourceTag",
        "AccessionNumber"
    ]]
)

# 4. See how many observations each metric has per year
print("\nRECORD COUNTS BY YEAR AND METRIC")
display(
    clean.groupby(["Year", "Metric"])
    .size()
    .unstack(fill_value=0)
)

print("\nDiagnostic check complete.")

YEAR COVERAGE BY COMPANY AND METRIC


,Company,Metric,min,max,nunique
0,Bank of America,CashAndCashEquivalents,2006,2019,14
1,Bank of America,NetIncome,2007,2025,19
2,Bank of America,Revenue,2007,2025,19
3,Bank of America,TotalAssets,2008,2025,18
4,Bank of America,TotalLiabilities,2008,2025,18
5,Citigroup,CashAndCashEquivalents,2015,2018,4
6,Citigroup,NetIncome,2007,2025,19
7,Citigroup,Revenue,2007,2025,19
8,Citigroup,TotalAssets,2008,2025,18
9,Citigroup,TotalLiabilities,2008,2025,18



REVENUE SOURCE TAGS IN RAW DATA


,Company,SourceTag,Records
0,Bank of America,Revenues,226
1,Citigroup,Revenues,311
2,JPMorgan Chase,Revenues,109
3,Morgan Stanley,RevenueFromContractWithCustomerExcludingAssess...,1
4,Wells Fargo,Revenues,37



ZERO-ASSETS RECORD IN RAW DATA


,Company,ReportedValueUSD,PeriodStart,PeriodEnd,FilingDate,Form,FiscalYear,FiscalPeriod,SourceTag,AccessionNumber
623,Bank of America,0,NaN,2008-12-31,2011-02-25,10-K,2010,FY,Assets,0000950123-11-018743



RECORD COUNTS BY YEAR AND METRIC


Metric,CashAndCashEquivalents,NetIncome,Revenue,TotalAssets,TotalLiabilities
Year,,,,,
2006,2,0,0,0,0
2007,2,6,3,0,0
2008,3,10,3,8,6
2009,2,12,6,6,6
2010,3,12,6,6,6
2011,3,12,6,6,6
2012,3,12,6,6,6
2013,3,12,6,6,6
2014,3,12,6,6,6



Diagnostic check complete.


In [5]:
import pandas as pd

df = pd.read_csv("sec_financial_data_clean_annual.csv")

df["PeriodEnd"] = pd.to_datetime(df["PeriodEnd"], errors="coerce")
df["ReportedValueUSD"] = pd.to_numeric(
    df["ReportedValueUSD"], errors="coerce"
)

# Start with the three metrics that have better coverage
selected_metrics = ["TotalAssets", "TotalLiabilities", "NetIncome"]

subset = df[df["Metric"].isin(selected_metrics)].copy()

# Flag implausible asset records for review; do not alter the raw data
zero_assets = subset[
    (subset["Metric"] == "TotalAssets") &
    (subset["ReportedValueUSD"] <= 0)
]

print("ASSET RECORDS REQUIRING REVIEW")
display(
    zero_assets[
        ["Company", "PeriodEnd", "FilingDate",
         "FiscalYear", "SourceTag", "ReportedValueUSD"]
    ]
)

# Reshape so each company-period has one column per metric
wide = subset.pivot_table(
    index=["Company", "PeriodEnd"],
    columns="Metric",
    values="ReportedValueUSD",
    aggfunc="last"
).reset_index()

wide.columns.name = None

# Calculate ratios only where the denominator is positive
wide["LiabilitiesToAssets"] = (
    wide["TotalLiabilities"] /
    wide["TotalAssets"].where(wide["TotalAssets"] > 0)
)

wide["NetIncomeToAssets"] = (
    wide["NetIncome"] /
    wide["TotalAssets"].where(wide["TotalAssets"] > 0)
)

wide = wide.sort_values(["Company", "PeriodEnd"])

wide.to_csv("sec_financial_indicators_initial.csv", index=False)

print("\nINITIAL FINANCIAL INDICATORS")
print("Rows:", len(wide))
print("Companies:", wide["Company"].nunique())
print("Period range:", wide["PeriodEnd"].min(), "to", wide["PeriodEnd"].max())
print("\nMissing values:")
print(wide.isnull().sum())

print("\nPreview:")
display(wide.head(12))

print("\nSaved: sec_financial_indicators_initial.csv")

ASSET RECORDS REQUIRING REVIEW


,Company,PeriodEnd,FilingDate,FiscalYear,SourceTag,ReportedValueUSD
7,Bank of America,2008-12-31,2011-02-25,2010,Assets,0



INITIAL FINANCIAL INDICATORS
Rows: 193
Companies: 6
Period range: 2007-11-30 00:00:00 to 2025-12-31 00:00:00

Missing values:
Company                 0
PeriodEnd               0
NetIncome               0
TotalAssets            83
TotalLiabilities       85
LiabilitiesToAssets    86
NetIncomeToAssets      84
dtype: int64

Preview:


,Company,PeriodEnd,NetIncome,TotalAssets,TotalLiabilities,LiabilitiesToAssets,NetIncomeToAssets
0,Bank of America,2007-12-31,1.498200e+10,NaN,NaN,NaN,NaN
1,Bank of America,2008-12-31,4.008000e+09,0.000000e+00,1.640891e+12,NaN,NaN
2,Bank of America,2009-12-31,6.276000e+09,2.330671e+12,2.105497e+12,0.903387,0.002693
3,Bank of America,2010-12-31,-2.238000e+09,2.264909e+12,2.036661e+12,0.899224,-0.000988
4,Bank of America,2011-12-31,1.446000e+09,2.129046e+12,1.898945e+12,0.891923,0.000679
5,Bank of America,2012-12-31,4.188000e+09,2.209974e+12,1.973018e+12,0.892779,0.001895
6,Bank of America,2013-12-31,1.143100e+10,2.102273e+12,1.869588e+12,0.889317,0.005437
7,Bank of America,2014-12-31,5.520000e+09,2.104534e+12,1.861063e+12,0.884311,0.002623
8,Bank of America,2015-12-31,1.591000e+10,2.144287e+12,1.888111e+12,0.880531,0.007420
9,Bank of America,2016-12-31,1.782200e+10,2.188067e+12,1.921872e+12,0.878342,0.008145



Saved: sec_financial_indicators_initial.csv


In [6]:
import pandas as pd

df = pd.read_csv("sec_financial_indicators_initial.csv")

metrics = ["NetIncome", "TotalAssets", "TotalLiabilities"]

# Count available metrics for each company-year
df["AvailableMetrics"] = df[metrics].notna().sum(axis=1)

print("COMPLETE COMPANY-YEAR RECORDS")
print("All 3 metrics available:", (df["AvailableMetrics"] == 3).sum())
print("Only 2 metrics available:", (df["AvailableMetrics"] == 2).sum())
print("Only 1 metric available:", (df["AvailableMetrics"] == 1).sum())

print("\nRECORDS MISSING FINANCIAL METRICS")
display(
    df.loc[
        df["AvailableMetrics"] < 3,
        ["Company", "PeriodEnd"] + metrics + ["AvailableMetrics"]
    ].head(25)
)

print("\nCOMPLETE RECORDS BY COMPANY")
display(
    df.assign(Complete=df["AvailableMetrics"] == 3)
      .groupby("Company")["Complete"]
      .agg(["sum", "count"])
      .rename(columns={"sum": "CompleteYears", "count": "TotalYears"})
)

print("\nZERO OR NEGATIVE ASSETS")
display(
    df[df["TotalAssets"].notna() & (df["TotalAssets"] <= 0)]
    [["Company", "PeriodEnd", "TotalAssets", "TotalLiabilities"]]
)

COMPLETE COMPANY-YEAR RECORDS
All 3 metrics available: 108
Only 2 metrics available: 2
Only 1 metric available: 83

RECORDS MISSING FINANCIAL METRICS


,Company,PeriodEnd,NetIncome,TotalAssets,TotalLiabilities,AvailableMetrics
0,Bank of America,2007-12-31,1.498200e+10,NaN,NaN,1
19,Citigroup,2007-12-31,3.617000e+09,NaN,NaN,1
21,Citigroup,2009-03-31,1.593000e+09,NaN,NaN,1
22,Citigroup,2009-06-30,4.279000e+09,NaN,NaN,1
23,Citigroup,2009-09-30,1.010000e+08,NaN,NaN,1
25,Citigroup,2010-03-31,4.428000e+09,NaN,NaN,1
26,Citigroup,2010-06-30,2.697000e+09,NaN,NaN,1
27,Citigroup,2010-09-30,2.168000e+09,NaN,NaN,1
29,Citigroup,2011-03-31,2.999000e+09,NaN,NaN,1
30,Citigroup,2011-06-30,3.341000e+09,NaN,NaN,1



COMPLETE RECORDS BY COMPANY


,CompleteYears,TotalYears
Company,,
Bank of America,18,19
Citigroup,18,58
Goldman Sachs,18,22
JPMorgan Chase,18,19
Morgan Stanley,18,56
Wells Fargo,18,19



ZERO OR NEGATIVE ASSETS


,Company,PeriodEnd,TotalAssets,TotalLiabilities
1,Bank of America,2008-12-31,0.0,1.640891e+12


In [7]:
import pandas as pd

df = pd.read_csv("sec_financial_indicators_initial.csv")
df["PeriodEnd"] = pd.to_datetime(df["PeriodEnd"], errors="coerce")

# Keep company-years with all three required financial metrics
metrics = ["NetIncome", "TotalAssets", "TotalLiabilities"]
complete = df.dropna(subset=metrics).copy()

# Exclude non-positive assets from this analysis copy
# The original record remains in the source dataset for investigation
analysis = complete[complete["TotalAssets"] > 0].copy()

# Add financial ratios
analysis["LiabilitiesToAssets"] = (
    analysis["TotalLiabilities"] / analysis["TotalAssets"]
)

analysis["NetIncomeToAssets"] = (
    analysis["NetIncome"] / analysis["TotalAssets"]
)

analysis["Year"] = analysis["PeriodEnd"].dt.year

analysis = analysis.sort_values(["Company", "PeriodEnd"])

analysis.to_csv("sec_financial_analysis_dataset.csv", index=False)

print("ANALYSIS DATASET")
print("Rows:", len(analysis))
print("Companies:", analysis["Company"].nunique())
print("Years:", analysis["Year"].min(), "-", analysis["Year"].max())

print("\nRows per company:")
print(analysis.groupby("Company").size())

print("\nMissing values:")
print(analysis[metrics + ["LiabilitiesToAssets", "NetIncomeToAssets"]].isna().sum())

print("\nSummary statistics:")
display(
    analysis[
        ["NetIncome", "TotalAssets", "TotalLiabilities",
         "LiabilitiesToAssets", "NetIncomeToAssets"]
    ].describe().round(4)
)

print("\nPreview:")
display(analysis.head(10))

print("\nSaved: sec_financial_analysis_dataset.csv")

from google.colab import files
files.download("sec_financial_analysis_dataset.csv")

ANALYSIS DATASET
Rows: 107
Companies: 6
Years: 2008 - 2025

Rows per company:
Company
Bank of America    17
Citigroup          18
Goldman Sachs      18
JPMorgan Chase     18
Morgan Stanley     18
Wells Fargo        18
dtype: int64

Missing values:
NetIncome              0
TotalAssets            0
TotalLiabilities       0
LiabilitiesToAssets    0
NetIncomeToAssets      0
dtype: int64

Summary statistics:


,NetIncome,TotalAssets,TotalLiabilities,LiabilitiesToAssets,NetIncomeToAssets
count,1.070000e+02,1.070000e+02,1.070000e+02,107.0000,107.0000
mean,1.509497e+10,1.869656e+12,1.695131e+12,0.9069,0.0077
std,1.234710e+10,8.217368e+11,7.496612e+11,0.0130,0.0044
min,-2.768400e+10,6.767640e+11,6.273080e+11,0.8711,-0.0143
25%,7.436500e+09,1.184186e+12,1.080271e+12,0.8987,0.0055
50%,1.365900e+10,1.895883e+12,1.701673e+12,0.9083,0.0085
75%,2.169000e+10,2.310836e+12,2.088962e+12,0.9164,0.0102
max,5.847100e+10,4.424900e+12,4.062462e+12,0.9316,0.0158



Preview:


,Company,PeriodEnd,NetIncome,TotalAssets,TotalLiabilities,LiabilitiesToAssets,NetIncomeToAssets,Year
2,Bank of America,2009-12-31,6.276000e+09,2.330671e+12,2.105497e+12,0.903387,0.002693,2009
3,Bank of America,2010-12-31,-2.238000e+09,2.264909e+12,2.036661e+12,0.899224,-0.000988,2010
4,Bank of America,2011-12-31,1.446000e+09,2.129046e+12,1.898945e+12,0.891923,0.000679,2011
5,Bank of America,2012-12-31,4.188000e+09,2.209974e+12,1.973018e+12,0.892779,0.001895,2012
6,Bank of America,2013-12-31,1.143100e+10,2.102273e+12,1.869588e+12,0.889317,0.005437,2013
7,Bank of America,2014-12-31,5.520000e+09,2.104534e+12,1.861063e+12,0.884311,0.002623,2014
8,Bank of America,2015-12-31,1.591000e+10,2.144287e+12,1.888111e+12,0.880531,0.007420,2015
9,Bank of America,2016-12-31,1.782200e+10,2.188067e+12,1.921872e+12,0.878342,0.008145,2016
10,Bank of America,2017-12-31,1.823200e+10,2.281234e+12,2.014088e+12,0.882894,0.007992,2017
11,Bank of America,2018-12-31,2.814700e+10,2.354507e+12,2.089182e+12,0.887312,0.011955,2018



Saved: sec_financial_analysis_dataset.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>